# Avance 3 · Baseline de regresión de la potencia (fase 2)

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este cuaderno es el compañero del [03_Baseline](03_Baseline.ipynb) para la fase 2. El baseline del Avance 3 clasificaba el par compuesto-diana en activo o inactivo a 10 µM; después la variable objetivo cambió: lo que se predice es la **potencia** (`pActividad`) y la clase se deriva de ella al final ([2.9](../docs/decisiones.md#d2-9)). Así que el baseline también es una regresión, y esto es lo que lo establece.

Las preguntas de la rúbrica del Avance 3, aplicadas aquí:

- **¿Qué algoritmo?** Tres, que se distinguen por lo que ven: el promedio (el piso), una recta con las mediciones exactas y **la misma recta con las censuradas como cotas** (regresión de Tobit).
- **¿Qué características pesan?** Los coeficientes de la recta, con y sin censura, y si coinciden.
- **¿Sobreajusta?** El error en entrenamiento contra el de validación.
- **¿Qué métrica?** La concordancia de Harrell, que usa también las censuradas; Spearman y RMSE solo con las exactas, declarado así.
- **¿Qué desempeño mínimo?** No hay un histórico. El piso es el promedio; el techo, el ruido propio de la medición.


## 1. Preparación

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo: la columna de tiempos es comparable

import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import scipy

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"

from src.evaluation.regresion import RUIDO_EXPERIMENTAL_IC50, RUIDO_EXPERIMENTAL_KI
from src.models.baseline_regresion import (ALFA_RIDGE, baselines, evaluar_baselines,
                                           intervalo_util, tabla)

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

sistema  Linux-6.6.114.1-microsoft-standard-WSL2-x86_64-with-glibc2.43
python   3.12.13
numpy    2.5.4
pandas   3.0.6
sklearn  1.9.1
scipy    1.18.1


### Los datos

La unidad es el par compuesto-diana, con los mismos pares de entrenamiento y validación que usa el [04](04_Models.ipynb) (la prueba reservada no se toca) y la diana como columnas de 0 y 1. Lo que cambia respecto de la clasificación es qué filas entran: **ninguna se descarta por no decidir a 10 µM**, porque una cota como `> 1 µM` no decide a 10 µM pero sí informa a un modelo con censura. Salen solo las filas que no informan nada: las que no tienen ninguna cota y las de cotas que se contradicen (intervalo vacío).

In [2]:
pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d = pares[pares["particion"].ne("prueba")].merge(desc, on="smiles").reset_index(drop=True)
cols = [c for c in desc.columns if c != "smiles"]
dianas = pd.get_dummies(d["target_gene"])

inferior = d["limite_inferior"].to_numpy(float)
superior = d["limite_superior"].to_numpy(float)
util = intervalo_util(inferior, superior)
print(f"{len(d):,} pares de entrenamiento y validación · {(~util).sum():,} sin información "
      f"(sin cota o con cotas cruzadas) y salen")

d, dianas = d[util].reset_index(drop=True), dianas[util].reset_index(drop=True)
inferior, superior = inferior[util], superior[util]
X = np.hstack([d[cols].to_numpy("float32"), dianas.to_numpy("float32")])
nombres = cols + list(dianas.columns)
exacta = inferior == superior
clase = d["activo_10um"].to_numpy(float)          # la clase derivada a 10 µM; NaN si no decide
grupos = d["grupo_cv"].to_numpy()
procedencia = d["estrato"].to_numpy()             # solo_chembl, solo_npass, mixto
print(f"{len(d):,} pares · {exacta.sum():,} con potencia exacta ({exacta.mean():.1%}) · "
      f"{(~exacta).sum():,} solo con cotas · {np.isnan(clase).sum():,} no deciden a 10 µM")

56,411 pares de entrenamiento y validación · 170 sin información (sin cota o con cotas cruzadas) y salen


56,241 pares · 46,100 con potencia exacta (82.0%) · 10,141 solo con cotas · 1,880 no deciden a 10 µM


## 2. Tres baselines

| Baseline | Qué ve | Para qué |
|:--|:--|:--|
| `media` | las exactas; predice siempre su promedio | el piso: un modelo que no le gane no aprendió nada |
| `ridge_exactas` | solo las exactas | la regresión lineal que un baseline sencillo haría |
| `tobit` | las exactas **y** las censuradas como cotas | la misma recta que Ridge, con la censura |

`ridge_exactas` y `tobit` son la misma familia con la misma penalización (`alfa = 100` de Ridge, escrita en la escala de cada uno), así que si difieren es por lo que cada uno ve y no por el modelo. El Tobit supone que la potencia es normal alrededor de la recta y maximiza la verosimilitud de lo que cada fila sí dice: una exacta aporta su valor y un `> 10 µM` la probabilidad de caer por debajo de pActividad 5 ([`src/models/tobit.py`](../src/models/tobit.py)).

La validación es la del Avance 3: cinco pliegues agrupados por `grupo_cv` (andamio y esqueleto de conectividad), semillas 0, 1 y 2. Las métricas de regresión usan el intervalo completo; MCC y AUROC se calculan sobre la clase derivada, cortando la potencia predicha en 5 (= 10 µM), para poder compararlas con el baseline de clasificación. Un estrato se reporta solo si tiene 20 casos de cada clase.

In [3]:
res = evaluar_baselines(X, inferior, superior, grupos, procedencia, clase)
print(f"{len(res):,} filas: {res['modelo'].nunique()} modelos × 3 semillas × 5 pliegues, por estrato")

180 filas: 3 modelos × 3 semillas × 5 pliegues, por estrato


## 3. Resultados

In [4]:
tabla_todos = tabla(res)
tabla_todos

,concordancia,desv_concordancia,spearman,rmse,mcc,auroc,rmse_entrenamiento,seg
modelo,,,,,,,,
tobit,0.700,0.007,0.512,1.234,0.362,0.792,1.181,9.276
ridge_exactas,0.692,0.006,0.530,1.156,0.225,0.776,1.115,1.381
media,0.500,0.000,NaN,1.337,0.000,0.500,1.338,0.723


### Por procedencia

Un promedio puede esconder cómo le va a los naturales: el control del Avance 3 mostró que el modelo separa mucho peor los pares que solo tiene NPASS.

In [5]:
por_procedencia = {e: tabla(res, e) for e in ("solo_chembl", "mixto", "solo_npass")
                   if (res["estrato"] == e).any()}
pd.concat(por_procedencia, names=["procedencia"])[["concordancia", "spearman", "rmse", "mcc", "auroc"]]

concordancia  spearman   rmse    mcc  auroc
procedencia modelo                                                    
solo_chembl tobit                 0.688     0.488  1.215  0.326  0.780
            ridge_exactas         0.681     0.503  1.148  0.172  0.762
            media                 0.500       NaN  1.325  0.000  0.500
mixto       tobit                 0.530     0.015  1.184  0.020  0.509
            ridge_exactas         0.510     0.043  1.092 -0.002  0.500
            media                 0.500       NaN  1.498  0.000  0.500
solo_npass  tobit                 0.639     0.192  1.514  0.114  0.624
            ridge_exactas         0.532     0.240  1.296  0.013  0.528
            media                 0.500       NaN  1.640  0.000  0.500

### ¿Sobreajusta?

El error en las exactas de entrenamiento contra el de validación. Una brecha grande diría que el modelo memoriza; un modelo lineal con penalización no debería.

In [6]:
t = tabla_todos[["rmse_entrenamiento", "rmse"]].rename(columns={"rmse": "rmse_validacion"})
t["brecha"] = (t["rmse_validacion"] - t["rmse_entrenamiento"]).round(3)
t

,rmse_entrenamiento,rmse_validacion,brecha
modelo,,,
tobit,1.181,1.234,0.053
ridge_exactas,1.115,1.156,0.041
media,1.338,1.337,-0.001


### Qué características pesan

Los coeficientes de la recta, sobre entradas estandarizadas, ajustada con todo entrenamiento y validación. Si la censura cambiara qué importa, los coeficientes de Ridge y Tobit diferirían; se comparan las quince entradas de mayor peso absoluto.

In [7]:
ridge = baselines()["ridge_exactas"].fit(X, inferior, superior)
tobit = baselines()["tobit"].fit(X, inferior, superior)
coef = pd.DataFrame({"ridge": ridge.modelo[-1].coef_, "tobit": tobit.pesos_}, index=nombres)

top = {k: set(coef[k].abs().nlargest(15).index) for k in coef}
print(f"de las 15 entradas de mayor peso, {len(top['ridge'] & top['tobit'])} coinciden entre Ridge y Tobit")
print(f"correlación de los coeficientes: {coef['ridge'].corr(coef['tobit']):.3f}")
print(f"σ del Tobit (ruido que atribuye a la medición): {tobit.sigma_:.2f} unidades de pActividad")
coef.reindex(coef["tobit"].abs().sort_values(ascending=False).index).head(15).round(3)

de las 15 entradas de mayor peso, 9 coinciden entre Ridge y Tobit
correlación de los coeficientes: 0.900
σ del Tobit (ruido que atribuye a la medición): 1.31 unidades de pActividad


,ridge,tobit
BertzCT,-0.360,-0.449
NumHAcceptors,0.287,0.429
HallKierAlpha,-0.338,-0.403
Chi3v,0.308,0.385
SMR_VSA3,-0.308,-0.366
Chi2n,-0.342,-0.339
SlogP_VSA6,-0.390,-0.337
MAOA,-0.172,-0.294
fr_NH1,0.374,0.289
LRRK2,0.183,0.254


### El piso y el techo

El **piso** es `media`. El **techo** es el ruido propio de la medición: repetir el mismo ensayo da diferencias de unas 0.7 unidades logarítmicas en IC50 y 0.5 en Ki (`kalliokoski2013ic50`, `kramer2012ki`; con curación mínima el ruido es mayor, `landrum2024ruido`). Un RMSE muy por debajo de eso no sería un buen resultado sino una señal de fuga.

In [8]:
print(f"techo de reproducibilidad: {RUIDO_EXPERIMENTAL_IC50} (IC50) y {RUIDO_EXPERIMENTAL_KI} (Ki) unidades logarítmicas")
tabla_todos[["rmse"]].assign(por_encima_del_techo_ic50=lambda t: (t["rmse"] > RUIDO_EXPERIMENTAL_IC50))

techo de reproducibilidad: 0.7 (IC50) y 0.5 (Ki) unidades logarítmicas


,rmse,por_encima_del_techo_ic50
modelo,,
tobit,1.234,True
ridge_exactas,1.156,True
media,1.337,True


## 4. Qué queda establecido, y qué no

**Lo que establece.**

1. **Hay señal más allá del promedio, y es modesta.** Los dos baselines lineales ordenan mejor que el azar: concordancia de 0.692 (Ridge) y 0.700 (Tobit) contra 0.500 de `media`, y AUROC de la clase derivada de 0.776 y 0.792. Pero el RMSE sobre las exactas es de 1.16 y 1.23 unidades de pActividad contra 1.34 del promedio: el error es 1.7 a 1.8 veces el ruido propio de la medición (0.7), lejos del techo y sin ninguna señal de fuga.
2. **La censura cambia qué tan bien se ordena la clase y cuesta exactitud en las exactas.** El Tobit supera a Ridge en MCC de la clase derivada (0.362 contra 0.225) y en AUROC, pero queda peor en Spearman (0.512 contra 0.530) y en RMSE (1.234 contra 1.156). Una lectura probable, que este cuaderno no demuestra: Ridge ve solo las exactas, que son mayormente potentes, y tiende a predecir «activo»; las cotas del Tobit lo corrigen hacia abajo.
3. **La diferencia está sobre todo en NPASS.** En los pares que solo tiene NPASS la concordancia es 0.639 con Tobit contra 0.532 con Ridge, que queda cerca del azar; en los de solo ChEMBL es 0.688 contra 0.681, y en los mixtos ninguno supera 0.53. Es también el estrato con menos pares.
4. **No sobreajustan.** La brecha entre el RMSE de entrenamiento y el de validación es de 0.04 a 0.05.
5. **Las dos rectas coinciden en lo que pesa:** los coeficientes correlacionan 0.90 y nueve de las quince entradas de mayor peso son las mismas. El σ que el Tobit atribuye a la medición, 1.31, es mayor que el techo de 0.7: dice cuánto deja sin explicar la recta, no que la medición sea tan ruidosa.
6. **El Tobit tarda unas siete veces más que Ridge** (9.3 s contra 1.4 s por ajuste). Esos tiempos se midieron con otra ejecución corriendo en la misma máquina: sirven para el orden, no como cifra.

**El desempeño mínimo para lo que sigue.** No hay un histórico contra el cual comparar. Los modelos del 04b tendrán que superar el mejor de cada métrica entre estos baselines: concordancia 0.700, Spearman 0.530 y RMSE 1.156 sobre las exactas.

**Lo que no establece.**

- **No es el modelo final.** La comparación de familias es del 04b.
- **La diferencia de concordancia entre Tobit y Ridge (0.008) es del orden de la dispersión entre evaluaciones** (0.006 y 0.007): por esa métrica no se distinguen. Sí se distinguen por MCC y en `solo_npass`.
- **El RMSE y el Spearman miran solo las exactas**, así que no dicen nada de qué tan bien se predicen las censuradas; para eso está la concordancia.
- **El MCC y el AUROC no son comparables con los del baseline de clasificación del [03](03_Baseline.ipynb):** son otro modelo, y aquí se calculan sobre los pares que deciden a 10 µM, con una potencia predicha cortada en 5.
- **Una sola partición de datos y tres semillas que comparten filas:** la dispersión que se reporta no es un error estándar.


In [9]:
# Registro en experimentos/ (decisión 2.9: la potencia como variable principal).
from src.tracking import Corrida

with Corrida("baseline-regresion", decision="2.9",
             params={"modelos": list(baselines()), "pliegues": 5, "alfa_ridge": ALFA_RIDGE,
                     "objetivo": "pActividad, con las censuradas como cotas (Tobit)",
                     "corte_mcc": 5.0, "estratos": "procedencia"},
             semillas=[0, 1, 2],
             datos=[PROCESADOS / "fase2_pares.parquet",
                    PROCESADOS / "fase2_descriptores_limpios.parquet"]) as c:
    c.metricas({"todos": tabla_todos.to_dict(orient="index"),
                **{e: t.to_dict(orient="index") for e, t in por_procedencia.items()}})

print(c.carpeta.name)

2026-10-25-ingrid-2.9-baseline-regresion


# Declaración de uso de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Escribir el módulo del baseline (`src/models/baseline_regresion.py`), el modelo de Tobit (`src/models/tobit.py`), las métricas de regresión (`src/evaluation/regresion.py`), sus pruebas y este cuaderno | Todas | Pruebas automáticas del gradiente, del signo de la censura y de los baselines (`tests/test_tobit.py`, `tests/test_regresion.py`, `tests/test_baseline_regresion.py`); la lectura de la sección 4 y la ejecución del cuaderno las revisa Ingrid Pamela Ruiz Puga (por confirmar en la revisión del PR) |

Las salidas de la herramienta se usaron como insumo de trabajo y el equipo asume la responsabilidad por la exactitud, la integridad y el contenido final del cuaderno.
